# Leaderboard Proxy README-Compatible Adapter Eval

この Notebook は `baseline/cot/phase0_offline_eval/infer_rule_based_adapter_phase0_offline_eval.ipynb` を最小変更で複製し、`cuda-train-data-analysis-v1/proof_first_solver_factory_routing/build_leaderboard_proxy_dataset.py` の **leaderboard 再現 proxy benchmark** を組み込んだものです。

- 推論エンジン: vLLM
- LoRA 読み込み: `LoRARequest`
- 推論条件: `README.md` の実評価値に固定
- 抽出 / 照合: `nvidia-nemotron-metric.ipynb` に合わせ、boxed-first 抽出と binary strict string compare を採用
- 目的: Kaggle 上で `leaderboard_proxy_v1_set` 200 問を実際に推論・採点し、LB 寄りの submit proxy として使う
- 入力データ: 事前生成済み leaderboard proxy artifacts を優先し、無ければ v3f / current の row-level artifacts から notebook 内で再構築する
- 注意: この benchmark は本物の hidden LB ではなく、v3f=0.7150 / current=0.5600 の差をローカル再現する engineered proxy です


In [ ]:
!pip install -U --no-index --find-links /kaggle/input/datasets/luciankucera/vllm-offline-wheels torch vllm datasets


In [ ]:
import polars as pl
import os
from vllm import LLM, SamplingParams
import torch
from datasets import Dataset
import re
import csv
import json
import math
import hashlib
from collections import Counter, defaultdict
from pathlib import Path

import kagglehub
import pandas as pd

torch.__version__


In [ ]:
import os
import re
from pathlib import Path

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
ADAPTER_PATH = "/kaggle/input/notebooks/renta0426/infer-rule-based-adapter-readme-compatibl/adapter"

PREBUILT_LEADERBOARD_PROXY_ROOT = "/kaggle/input/datasets/renta0426/leaderboard-proxy-v1-artifacts"
PREBUILT_LEADERBOARD_PROXY_DATASET_CSV = f"{PREBUILT_LEADERBOARD_PROXY_ROOT}/leaderboard_proxy_v1_set.csv"
PREBUILT_LEADERBOARD_PROXY_SUMMARY_JSON = f"{PREBUILT_LEADERBOARD_PROXY_ROOT}/leaderboard_proxy_v1_summary.json"
PREBUILT_LEADERBOARD_PROXY_REPORT_MD = f"{PREBUILT_LEADERBOARD_PROXY_ROOT}/leaderboard_proxy_v1_report.md"

SOURCE_PHASE0_SET_CSV = "/kaggle/input/datasets/renta0426/phase0-offline-eval-artifacts/phase0_combined_eval_set.csv"
SOURCE_SPECIALIZED_SET_CSV = "/kaggle/input/datasets/renta0426/binary-bias-specialized-eval-artifacts/binary_bias_specialized_set.csv"
V3F_PHASE0_ROW_LEVEL_CSV = "/kaggle/input/datasets/renta0426/v3f-phase0-offline-eval-artifacts/phase0_eval_row_level.csv"
CURRENT_PHASE0_ROW_LEVEL_CSV = "/kaggle/input/datasets/renta0426/proof-first-phase0-offline-eval-artifacts/phase0_eval_row_level.csv"
V3F_SPECIALIZED_ROW_LEVEL_CSV = "/kaggle/input/datasets/renta0426/v3f-binary-bias-specialized-eval-artifacts/binary_bias_specialized_eval_row_level.csv"
CURRENT_SPECIALIZED_ROW_LEVEL_CSV = "/kaggle/input/datasets/renta0426/proof-first-binary-bias-specialized-eval-artifacts/binary_bias_specialized_eval_row_level.csv"

WORKING_ROOT = "/kaggle/working/leaderboard_proxy_eval"
WORKING_ARTIFACT_ROOT = f"{WORKING_ROOT}/artifacts"
WORKING_REPORT_ROOT = f"{WORKING_ROOT}/reports"
WORKING_COMBINED_BENCHMARK_CSV = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_v1_set.csv"
WORKING_PROXY_SUMMARY_JSON = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_v1_summary.json"
WORKING_PROXY_REPORT_MD = f"{WORKING_REPORT_ROOT}/leaderboard_proxy_v1_report.md"
WORKING_MANIFEST_JSON = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_eval_manifest.json"
WORKING_MANIFEST_MD = f"{WORKING_REPORT_ROOT}/leaderboard_proxy_eval_manifest.md"
WORKING_RAW_OUTPUTS_CSV = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_eval_raw_outputs.csv"
WORKING_ROW_LEVEL_CSV = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_eval_row_level.csv"
WORKING_SUMMARY_JSON = f"{WORKING_ARTIFACT_ROOT}/leaderboard_proxy_eval_summary.json"
WORKING_SUMMARY_MD = f"{WORKING_REPORT_ROOT}/leaderboard_proxy_eval_summary.md"

README_MAX_LORA_RANK = 32
README_MAX_TOKENS = 7680
README_TOP_P = 1.0
README_TEMPERATURE = 0.0
README_MAX_NUM_SEQS = 64
README_GPU_MEMORY_UTILIZATION = 0.85
README_MAX_MODEL_LEN = 8192

adapter_path = Path(ADAPTER_PATH)
required_files = [adapter_path / "adapter_config.json", adapter_path / "adapter_model.safetensors"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Adapter directory must contain adapter_config.json and adapter_model.safetensors. Missing: "
        + ", ".join(missing)
    )
adapter_config = json.loads((adapter_path / "adapter_config.json").read_text(encoding="utf-8"))
adapter_rank = int(adapter_config.get("r", -1))
if adapter_rank < 1 or adapter_rank > README_MAX_LORA_RANK:
    raise ValueError(
        f"Adapter rank {adapter_rank} exceeds README / metric max_lora_rank={README_MAX_LORA_RANK}."
    )

Path(WORKING_ARTIFACT_ROOT).mkdir(parents=True, exist_ok=True)
Path(WORKING_REPORT_ROOT).mkdir(parents=True, exist_ok=True)

print(f"MODEL_PATH={MODEL_PATH}")
print(f"ADAPTER_PATH={adapter_path}")
print(f"ADAPTER_RANK={adapter_rank}")
print(f"PREBUILT_LEADERBOARD_PROXY_ROOT={PREBUILT_LEADERBOARD_PROXY_ROOT}")
print(f"SOURCE_PHASE0_SET_CSV={SOURCE_PHASE0_SET_CSV}")
print(f"SOURCE_SPECIALIZED_SET_CSV={SOURCE_SPECIALIZED_SET_CSV}")


In [ ]:
GENERAL_STABLE_QUOTAS = {
    "gravity_constant": 50,
    "unit_conversion": 50,
    "roman_numeral": 50,
    "text_decryption": 50,
}
BINARY_HARD_TIER_QUOTAS = {
    "verified_trace_ready": 20,
    "answer_only_keep": 20,
    "manual_audit_priority": 20,
}
SYMBOL_WATCH_TARGETS = [
    ("numeric_2x2", "verified_trace_ready", 15),
    ("numeric_2x2", "answer_only_keep", 15),
    ("numeric_2x2", "manual_audit_priority", 10),
    ("glyph_len5", "manual_audit_priority", 20),
]
HOLDOUT_FOLDS = 5
BOXED_PATTERN = re.compile(r"\\boxed\{([^}]*)(?:\}|$)")
FINAL_ANSWER_PATTERNS = (
    r"The final answer is:\s*([^\n]+)",
    r"Final answer is:\s*([^\n]+)",
    r"Final answer\s*[:：]\s*([^\n]+)",
    r"final answer\s*[:：]\s*([^\n]+)",
)
NUMBER_PATTERN = re.compile(r"-?\d+(?:\.\d+)?")
BIT8_PATTERN = re.compile(r"^[01]{8}$")
METRIC_NOTEBOOK_DEFAULTS = {
    "max_lora_rank": 32,
    "max_tokens": 3584,
    "top_p": 1.0,
    "temperature": 1.0,
    "max_num_seqs": 128,
    "gpu_memory_utilization": 0.85,
    "max_model_len": 4096,
}
FAMILY_SHORT = {
    "gravity_constant": "gravity",
    "unit_conversion": "unit",
    "roman_numeral": "roman",
    "text_decryption": "text",
    "bit_manipulation": "binary",
    "symbol_equation": "symbol",
}


def parse_bool(value):
    return str(value).strip().lower() in {"1", "true", "yes"}


def parse_int(value, default=0):
    try:
        text = str(value).strip()
        if not text:
            return default
        return int(float(text))
    except (TypeError, ValueError):
        return default


def parse_float(value, default=None):
    try:
        text = str(value).strip()
        if not text:
            return default
        return float(text)
    except (TypeError, ValueError):
        return default


def stable_mod(text, mod):
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:16], 16) % mod


def load_csv_rows(path):
    with Path(path).open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f"Missing CSV header: {path}")
        return [dict(row) for row in reader]


def write_csv_rows(path, rows, fieldnames):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.rstrip() + "\n", encoding="utf-8")


def prompt_len_bucket(length):
    if length < 300:
        return "<300"
    if length < 400:
        return "300-399"
    if length < 500:
        return "400-499"
    if length < 600:
        return "500-599"
    return "600+"


def score_rank_low(row):
    hard_score = parse_float(row.get("hard_score"), 999.0)
    if hard_score is None:
        hard_score = 999.0
    return (
        hard_score,
        parse_int(row.get("prompt_len_chars"), 99999),
        -parse_int(row.get("num_examples"), 0),
        row.get("id", ""),
    )


def score_rank_high(row):
    hard_score = parse_float(row.get("hard_score"), -999.0)
    if hard_score is None:
        hard_score = -999.0
    return (
        -hard_score,
        -parse_int(row.get("prompt_len_chars"), 0),
        -parse_int(row.get("num_examples"), 0),
        row.get("id", ""),
    )


def balanced_take(rows, *, quota, group_keys, hard_first):
    grouped = defaultdict(list)
    for row in rows:
        key = tuple(str(row.get(name, "") or "") for name in group_keys)
        grouped[key].append(row)
    rank_fn = score_rank_high if hard_first else score_rank_low
    ordered_groups = sorted(grouped.items(), key=lambda item: (item[0], len(item[1])))
    for _, group_rows in ordered_groups:
        group_rows.sort(key=rank_fn)
    selected = []
    while len(selected) < quota:
        progressed = False
        for _, group_rows in ordered_groups:
            if not group_rows:
                continue
            selected.append(group_rows.pop(0))
            progressed = True
            if len(selected) >= quota:
                break
        if not progressed:
            break
    return selected


def normalize_family_label(row):
    family = row.get("family", "")
    if family in FAMILY_SHORT:
        return FAMILY_SHORT[family]
    label = row.get("label", "")
    if label:
        return label
    return family


def benchmark_columns():
    return [
        "benchmark_name",
        "benchmark_role",
        "benchmark_index",
        "id",
        "family",
        "family_short",
        "template_subtype",
        "selection_tier",
        "teacher_solver_candidate",
        "answer_type",
        "num_examples",
        "prompt_len_chars",
        "hard_score",
        "group_signature",
        "query_raw",
        "answer",
        "prompt",
    ]


def to_benchmark_row(row, *, benchmark_name, benchmark_role):
    return {
        "benchmark_name": benchmark_name,
        "benchmark_role": benchmark_role,
        "id": row.get("id", ""),
        "family": row.get("family", ""),
        "family_short": normalize_family_label(row),
        "template_subtype": row.get("template_subtype", ""),
        "selection_tier": row.get("selection_tier", ""),
        "teacher_solver_candidate": row.get("teacher_solver_candidate", ""),
        "answer_type": row.get("answer_type", ""),
        "num_examples": parse_int(row.get("num_examples"), 0),
        "prompt_len_chars": parse_int(row.get("prompt_len_chars"), 0),
        "hard_score": parse_float(row.get("hard_score"), 0.0),
        "group_signature": row.get("group_signature", ""),
        "query_raw": row.get("query_raw", ""),
        "answer": row.get("answer", ""),
        "prompt": row.get("prompt", ""),
    }


def build_general_stable_set(rows):
    selected = []
    for family, quota in GENERAL_STABLE_QUOTAS.items():
        candidates = [
            row
            for row in rows
            if row.get("family") == family
            and row.get("selection_tier") == "verified_trace_ready"
            and parse_bool(row.get("verified_trace_ready", "true"))
        ]
        family_rows = balanced_take(
            candidates,
            quota=quota,
            group_keys=("template_subtype", "teacher_solver_candidate"),
            hard_first=False,
        )
        selected.extend(
            to_benchmark_row(row, benchmark_name="general_stable_set", benchmark_role="stable_replay")
            for row in family_rows
        )
    return selected


def build_binary_hard_set(rows):
    selected = []
    binary_rows = [row for row in rows if row.get("family") == "bit_manipulation"]
    for tier, quota in BINARY_HARD_TIER_QUOTAS.items():
        candidates = [row for row in binary_rows if row.get("selection_tier") == tier]
        group_keys = (
            "teacher_solver_candidate",
            "bit_structured_formula_abstract_family",
            "group_signature",
        )
        tier_rows = balanced_take(candidates, quota=quota, group_keys=group_keys, hard_first=True)
        selected.extend(
            to_benchmark_row(row, benchmark_name="binary_hard_set", benchmark_role="hard_binary_watch")
            for row in tier_rows
        )
    return selected


def fill_symbol_watch_candidates(rows, already_selected_ids):
    remaining = [
        row
        for row in rows
        if row.get("family") == "symbol_equation" and row.get("id", "") not in already_selected_ids
    ]
    remaining.sort(key=score_rank_high)
    filler = []
    for row in remaining:
        filler.append(to_benchmark_row(row, benchmark_name="symbol_watch_set", benchmark_role="symbol_watch"))
        if len(filler) >= 60:
            break
    return filler


def build_symbol_watch_set(rows):
    selected = []
    selected_ids = set()
    symbol_rows = [row for row in rows if row.get("family") == "symbol_equation"]
    for template_subtype, tier, quota in SYMBOL_WATCH_TARGETS:
        candidates = [
            row
            for row in symbol_rows
            if row.get("template_subtype") == template_subtype and row.get("selection_tier") == tier
        ]
        watch_rows = balanced_take(
            candidates,
            quota=quota,
            group_keys=("symbol_query_operator", "teacher_solver_candidate"),
            hard_first=True,
        )
        for row in watch_rows:
            if row.get("id", "") in selected_ids:
                continue
            selected.append(to_benchmark_row(row, benchmark_name="symbol_watch_set", benchmark_role="symbol_watch"))
            selected_ids.add(row.get("id", ""))
    if len(selected) < 60:
        for row in fill_symbol_watch_candidates(rows, selected_ids):
            if row["id"] in selected_ids:
                continue
            selected.append(row)
            selected_ids.add(row["id"])
            if len(selected) >= 60:
                break
    return selected


def holdout_key_structured_family(row):
    value = str(row.get("bit_structured_formula_abstract_family", "")).strip()
    return value or "__none__"


def holdout_key_solver(row):
    value = str(row.get("teacher_solver_candidate", "")).strip()
    return value or "__none__"


def holdout_key_gap(row):
    num_examples = parse_int(row.get("num_examples"), 0)
    no_candidate = parse_int(row.get("bit_no_candidate_positions"), -1)
    multi_candidate = parse_int(row.get("bit_multi_candidate_positions"), -1)
    return f"ex{num_examples}__no{no_candidate}__multi{multi_candidate}"


def holdout_key_prompt_signature(row):
    group_signature = str(row.get("group_signature", "")).strip()
    if group_signature:
        return group_signature
    prompt = str(row.get("prompt", ""))
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]


def assign_balanced_group_folds(keys, num_folds):
    group_counts = Counter(keys)
    fold_loads = [0 for _ in range(num_folds)]
    assignments = {}
    ordered_groups = sorted(group_counts.items(), key=lambda item: (-item[1], item[0]))
    for group_key, group_size in ordered_groups:
        preferred = stable_mod(group_key, num_folds)
        best_fold = min(
            range(num_folds),
            key=lambda fold: (fold_loads[fold], abs(fold - preferred), fold),
        )
        assignments[group_key] = best_fold
        fold_loads[best_fold] += group_size
    return assignments


def build_binary_holdout_assignments(rows):
    binary_rows = [row for row in rows if row.get("family") == "bit_manipulation"]
    key_builders = {
        "structured_family": holdout_key_structured_family,
        "solver_family": holdout_key_solver,
        "gap_signature": holdout_key_gap,
        "prompt_signature": holdout_key_prompt_signature,
    }
    fold_maps = {
        holdout_kind: assign_balanced_group_folds([key_builder(row) for row in binary_rows], HOLDOUT_FOLDS)
        for holdout_kind, key_builder in key_builders.items()
    }
    assignments = []
    for row in binary_rows:
        for holdout_kind, key_builder in key_builders.items():
            holdout_key = key_builder(row)
            fold = fold_maps[holdout_kind][holdout_key]
            assignments.append(
                {
                    "id": row.get("id", ""),
                    "family": row.get("family", ""),
                    "selection_tier": row.get("selection_tier", ""),
                    "template_subtype": row.get("template_subtype", ""),
                    "teacher_solver_candidate": row.get("teacher_solver_candidate", ""),
                    "holdout_kind": holdout_kind,
                    "holdout_key": holdout_key,
                    "fold": fold,
                    "num_examples": parse_int(row.get("num_examples"), 0),
                    "bit_no_candidate_positions": parse_int(row.get("bit_no_candidate_positions"), -1),
                    "bit_multi_candidate_positions": parse_int(row.get("bit_multi_candidate_positions"), -1),
                    "group_signature": row.get("group_signature", ""),
                }
            )
    return assignments


def summarize_benchmark(rows):
    family_counts = Counter(row["family_short"] for row in rows)
    tier_counts = Counter(row["selection_tier"] for row in rows)
    template_counts = Counter(row["template_subtype"] for row in rows)
    return {
        "rows": len(rows),
        "family_counts": dict(sorted(family_counts.items())),
        "selection_tier_counts": dict(sorted(tier_counts.items())),
        "template_subtype_counts": dict(sorted(template_counts.items())),
    }


def build_manifest(*, analysis_csv, general_rows, binary_rows, symbol_rows, holdouts):
    holdout_counts = defaultdict(lambda: defaultdict(int))
    for row in holdouts:
        holdout_counts[row["holdout_kind"]][str(row["fold"])] += 1
    return {
        "phase": "phase0_offline_eval",
        "source_analysis_csv": str(analysis_csv),
        "readme_eval_assumptions": {
            "metric": "accuracy",
            "temperature": 0.0,
            "top_p": 1.0,
            "max_tokens": 7680,
            "max_num_seqs": 64,
            "max_model_len": 8192,
            "boxed_first_extraction": True,
            "binary_answers_are_string_exact": True,
            "numeric_relative_tolerance": 1e-2,
            "public_metric_notebook_defaults": METRIC_NOTEBOOK_DEFAULTS,
        },
        "benchmark_sets": {
            "general_stable_set": summarize_benchmark(general_rows),
            "binary_hard_set": summarize_benchmark(binary_rows),
            "symbol_watch_set": summarize_benchmark(symbol_rows),
        },
        "binary_holdouts": {
            holdout_kind: dict(sorted(folds.items()))
            for holdout_kind, folds in sorted(holdout_counts.items())
        },
    }


def render_phase0_report(manifest):
    lines = []
    lines.append("# Phase 0 Offline Eval Manifest")
    lines.append("")
    lines.append("## README-aligned evaluation assumptions")
    lines.append("")
    for key, value in manifest["readme_eval_assumptions"].items():
        lines.append(f"- {key}: `{value}`")
    lines.append("")
    lines.append("## Benchmark sets")
    lines.append("")
    lines.append("| set | rows | family_counts | selection_tier_counts |")
    lines.append("| --- | ---: | --- | --- |")
    for set_name, payload in manifest["benchmark_sets"].items():
        lines.append(
            f"| `{set_name}` | {payload['rows']} | `{json.dumps(payload['family_counts'], ensure_ascii=False)}` | `{json.dumps(payload['selection_tier_counts'], ensure_ascii=False)}` |"
        )
    lines.append("")
    lines.append("## Binary holdout fold counts")
    lines.append("")
    lines.append("| holdout_kind | fold_counts |")
    lines.append("| --- | --- |")
    for holdout_kind, fold_counts in manifest["binary_holdouts"].items():
        lines.append(f"| `{holdout_kind}` | `{json.dumps(fold_counts, ensure_ascii=False)}` |")
    lines.append("")
    return "\n".join(lines)


def mark_status_rows(rows, benchmark_name):
    marked = []
    for index, row in enumerate(rows, start=1):
        new_row = dict(row)
        new_row["benchmark_index"] = index
        new_row["benchmark_name"] = benchmark_name
        marked.append(new_row)
    return marked


def extract_final_answer(text):
    if text is None:
        return "NOT_FOUND"
    matches = BOXED_PATTERN.findall(text)
    if matches:
        non_empty = [match.strip() for match in matches if match.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    for pattern in FINAL_ANSWER_PATTERNS:
        matched = re.findall(pattern, text, re.IGNORECASE)
        if matched:
            return matched[-1].strip()
    numeric_matches = NUMBER_PATTERN.findall(text)
    if numeric_matches:
        return numeric_matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify_answer(gold, predicted):
    gold = str(gold).strip()
    predicted = str(predicted).strip()
    if re.fullmatch(r"[01]+", gold):
        return predicted.lower() == gold.lower()
    try:
        gold_num = float(gold)
        pred_num = float(predicted)
        return math.isclose(gold_num, pred_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == gold.lower()


def analyze_raw_output(text):
    if text is None:
        return {
            "extracted_answer": "NOT_FOUND",
            "fallback_type": "not_found",
            "format_bucket": "not_found",
            "has_boxed": False,
            "boxed_count": 0,
        }
    boxed_matches = BOXED_PATTERN.findall(text)
    numeric_matches = NUMBER_PATTERN.findall(text)
    if boxed_matches:
        non_empty = [match.strip() for match in boxed_matches if match.strip()]
        if non_empty:
            extracted = non_empty[-1]
            return {
                "extracted_answer": extracted,
                "fallback_type": "boxed_non_empty",
                "format_bucket": "boxed",
                "has_boxed": True,
                "boxed_count": len(boxed_matches),
            }
        extracted = boxed_matches[-1].strip()
        return {
            "extracted_answer": extracted,
            "fallback_type": "boxed_empty",
            "format_bucket": "boxed_empty",
            "has_boxed": True,
            "boxed_count": len(boxed_matches),
        }
    for pattern in FINAL_ANSWER_PATTERNS:
        matched = re.findall(pattern, text, re.IGNORECASE)
        if matched:
            return {
                "extracted_answer": matched[-1].strip(),
                "fallback_type": "final_answer_phrase",
                "format_bucket": "final_answer",
                "has_boxed": False,
                "boxed_count": 0,
            }
    if numeric_matches:
        return {
            "extracted_answer": numeric_matches[-1],
            "fallback_type": "last_number",
            "format_bucket": "numeric_fallback",
            "has_boxed": False,
            "boxed_count": 0,
        }
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return {
        "extracted_answer": lines[-1] if lines else "NOT_FOUND",
        "fallback_type": "last_line" if lines else "not_found",
        "format_bucket": "line_fallback" if lines else "not_found",
        "has_boxed": False,
        "boxed_count": 0,
    }


def safe_div(numerator, denominator):
    if denominator == 0:
        return 0.0
    return numerator / denominator


def aggregate_counts(rows, key):
    buckets = defaultdict(lambda: {"rows": 0, "correct": 0})
    for row in rows:
        bucket_key = str(row.get(key, ""))
        buckets[bucket_key]["rows"] += 1
        buckets[bucket_key]["correct"] += int(bool(row.get("is_correct")))
    summary = []
    for bucket_key, stats in sorted(buckets.items()):
        summary.append(
            {
                key: bucket_key,
                "rows": stats["rows"],
                "correct": stats["correct"],
                "accuracy": round(safe_div(stats["correct"], stats["rows"]), 4),
            }
        )
    return summary


def build_binary_metrics(rows):
    binary_rows = [row for row in rows if row.get("family") == "bit_manipulation" or row.get("family_short") == "binary"]
    regex_ok = [row for row in binary_rows if BIT8_PATTERN.fullmatch(str(row.get("prediction", "")))]
    gold_leading_zero_rows = [row for row in binary_rows if str(row.get("gold_answer", "")).startswith("0")]
    leading_zero_ok = [
        row
        for row in gold_leading_zero_rows
        if BIT8_PATTERN.fullmatch(str(row.get("prediction", ""))) and str(row.get("prediction", "")).startswith("0")
    ]
    format_ok = [
        row
        for row in binary_rows
        if row.get("has_boxed") and BIT8_PATTERN.fullmatch(str(row.get("prediction", "")))
    ]
    format_fail = [row for row in binary_rows if row not in format_ok]
    format_ok_but_wrong = [row for row in format_ok if not row.get("is_correct")]
    return {
        "rows": len(binary_rows),
        "boxed_extraction_success_rate": round(
            safe_div(sum(int(row.get("fallback_type") == "boxed_non_empty") for row in binary_rows), len(binary_rows)), 4
        ),
        "regex_exact_rate": round(safe_div(len(regex_ok), len(binary_rows)), 4),
        "leading_zero_retention_rate": round(safe_div(len(leading_zero_ok), len(gold_leading_zero_rows)), 4),
        "format_failure_rate": round(safe_div(len(format_fail), len(binary_rows)), 4),
        "format_ok_content_wrong_rate": round(safe_div(len(format_ok_but_wrong), len(format_ok)), 4),
        "solver_family_accuracy": aggregate_counts(binary_rows, "teacher_solver_candidate"),
    }


def summarize_scored_rows(row_level):
    return {
        "overall": {
            "rows": len(row_level),
            "correct": sum(int(row["is_correct"]) for row in row_level),
            "accuracy": round(safe_div(sum(int(row["is_correct"]) for row in row_level), len(row_level)), 4),
        },
        "by_family": aggregate_counts(row_level, "family_short"),
        "by_template_subtype": aggregate_counts(row_level, "template_subtype"),
        "by_answer_type": aggregate_counts(row_level, "answer_type"),
        "by_prompt_len_bucket": aggregate_counts(row_level, "prompt_len_bucket"),
        "by_num_examples": aggregate_counts(row_level, "num_examples"),
        "by_selection_tier": aggregate_counts(row_level, "selection_tier"),
        "binary_metrics": build_binary_metrics(row_level),
    }


def render_markdown_summary(name, summary):
    lines = []
    lines.append(f"# {name}")
    lines.append("")
    overall = summary["overall"]
    lines.append("## Overall")
    lines.append("")
    lines.append(f"- rows: `{overall['rows']}`")
    lines.append(f"- correct: `{overall['correct']}`")
    lines.append(f"- accuracy: `{overall['accuracy']:.4f}`")
    lines.append("")

    def add_table(title, rows, key_name):
        lines.append(f"## {title}")
        lines.append("")
        lines.append(f"| {key_name} | rows | correct | accuracy |")
        lines.append("| --- | ---: | ---: | ---: |")
        for row in rows:
            lines.append(f"| `{row[key_name]}` | {row['rows']} | {row['correct']} | {row['accuracy']:.4f} |")
        lines.append("")

    add_table("Family accuracy", summary["by_family"], "family_short")
    add_table("Template subtype accuracy", summary["by_template_subtype"], "template_subtype")
    add_table("Answer type accuracy", summary["by_answer_type"], "answer_type")
    add_table("Prompt length buckets", summary["by_prompt_len_bucket"], "prompt_len_bucket")
    add_table("Num examples", summary["by_num_examples"], "num_examples")
    add_table("Selection tier accuracy", summary["by_selection_tier"], "selection_tier")

    binary_metrics = summary["binary_metrics"]
    lines.append("## Binary metrics")
    lines.append("")
    for key in (
        "rows",
        "boxed_extraction_success_rate",
        "regex_exact_rate",
        "leading_zero_retention_rate",
        "format_failure_rate",
        "format_ok_content_wrong_rate",
    ):
        lines.append(f"- {key}: `{binary_metrics[key]}`")
    lines.append("")
    lines.append("### Binary solver-family accuracy")
    lines.append("")
    lines.append("| teacher_solver_candidate | rows | correct | accuracy |")
    lines.append("| --- | ---: | ---: | ---: |")
    for row in binary_metrics["solver_family_accuracy"]:
        lines.append(
            f"| `{row['teacher_solver_candidate']}` | {row['rows']} | {row['correct']} | {row['accuracy']:.4f} |"
        )
    lines.append("")
    return "\n".join(lines)


def build_prompts(tokenizer, prompt_series):
    prompts = []
    for prompt_text in prompt_series:
        user_content = (
            prompt_text
            + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
        )
        try:
            rendered = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            rendered = user_content
        prompts.append(rendered)
    return prompts


def build_binary_holdout_accuracy_rows(scored_rows, holdout_rows):
    binary_by_id = {
        row["id"]: row
        for row in scored_rows
        if row.get("family") == "bit_manipulation" or row.get("family_short") == "binary"
    }
    joined = []
    for holdout in holdout_rows:
        scored = binary_by_id.get(holdout.get("id", ""))
        if scored is None:
            continue
        joined.append(
            {
                "holdout_kind": holdout.get("holdout_kind", ""),
                "fold": str(holdout.get("fold", "")),
                "rows": 1,
                "correct": int(bool(scored.get("is_correct"))),
            }
        )
    buckets = defaultdict(lambda: {"rows": 0, "correct": 0})
    for row in joined:
        key = (row["holdout_kind"], row["fold"])
        buckets[key]["rows"] += row["rows"]
        buckets[key]["correct"] += row["correct"]
    summary = []
    for (holdout_kind, fold), stats in sorted(buckets.items()):
        summary.append(
            {
                "holdout_kind": holdout_kind,
                "fold": fold,
                "rows": stats["rows"],
                "correct": stats["correct"],
                "accuracy": round(safe_div(stats["correct"], stats["rows"]), 4),
            }
        )
    return summary


In [ ]:
LEADERBOARD_PROXY_TARGET_V3F = 0.715
LEADERBOARD_PROXY_TARGET_CURRENT = 0.56
LEADERBOARD_PROXY_TOTAL_ROWS = 200
LEADERBOARD_PROXY_NAME = "leaderboard_proxy_v1_set"


def load_proxy_row_level_rows(path):
    rows = {}
    for row in load_csv_rows(path):
        row["_corrected_is_correct"] = verify_answer(row["gold_answer"], row["prediction"])
        rows[row["id"]] = row
    return rows


def load_proxy_source_rows(phase0_path, specialized_path):
    merged = {}
    for source_name, path, source_split in (
        ("phase0_combined_eval_set", phase0_path, "phase0"),
        ("binary_bias_specialized_set", specialized_path, "specialized"),
    ):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required source set not found: {path}")
        for row in load_csv_rows(path):
            payload = dict(row)
            payload["_source_name"] = source_name
            payload["_source_split"] = source_split
            payload.setdefault("v1_focus_bucket", "")
            payload.setdefault("v1_exposure_band", "")
            payload.setdefault("v1_bucket_note", "")
            if not payload.get("family_short"):
                payload["family_short"] = normalize_family_label(payload)
            merged[row["id"]] = payload
    return merged


def proxy_role_of(v3f_correct, current_correct):
    if v3f_correct and current_correct:
        return "both_correct"
    if v3f_correct and not current_correct:
        return "v3f_only"
    if current_correct and not v3f_correct:
        return "current_only"
    return "both_wrong"


def proxy_bucket(row):
    value = str(row.get("v1_focus_bucket", "")).strip()
    if value:
        return value
    family_short = str(row.get("family_short", "")).strip() or normalize_family_label(row)
    subtype = str(row.get("template_subtype", "")).strip()
    if family_short and subtype:
        return f"{family_short}:{subtype}"
    return family_short or "unknown"


def proxy_source_priority(row):
    role = row["proxy_role"]
    focus = str(row.get("v1_focus_bucket", "")).strip()
    family_short = str(row.get("family_short", "")).strip()
    if role == "v3f_only":
        priorities = {
            "supported_bijection": 200,
            "dominant_structured_safe": 190,
            "supported_affine_xor": 180,
            "boolean_family": 170,
            "dominant_structured_abstract": 160,
            "no_solver_answer_only": 150,
            "no_solver_manual": 140,
            "rare_perm_independent": 130,
            "rare_byte_transform": 120,
        }
        return priorities.get(focus, 20 if family_short == "binary" else 10)
    if role == "both_correct":
        if family_short == "gravity":
            return 200
        if family_short == "roman":
            return 198
        if family_short == "unit":
            return 196
        if family_short == "text":
            return 194
        priorities = {
            "supported_bijection": 190,
            "dominant_structured_safe": 188,
            "supported_affine_xor": 186,
            "boolean_family": 184,
            "dominant_structured_abstract": 182,
            "rare_perm_independent": 180,
            "rare_byte_transform": 178,
            "no_solver_answer_only": 170,
            "numeric_2x2": 160,
        }
        return priorities.get(focus, 80 if family_short == "symbol" else 60)
    if role == "both_wrong":
        priorities = {
            "supported_not_structured": 220,
            "no_solver_manual": 210,
            "no_solver_answer_only": 200,
            "dominant_structured_abstract": 190,
            "boolean_family": 180,
            "supported_affine_xor": 170,
            "supported_bijection": 160,
            "dominant_structured_safe": 150,
            "glyph_len5": 140,
            "numeric_2x2": 130,
        }
        return priorities.get(focus, 120 if family_short == "binary" else 100)
    return 0


def proxy_rank_key(row):
    return (
        -proxy_source_priority(row),
        -parse_float(row.get("hard_score"), 0.0),
        -parse_int(row.get("prompt_len_chars"), 0),
        -parse_int(row.get("num_examples"), 0),
        row.get("id", ""),
    )


def proxy_round_robin_take(rows, quota, key_fn):
    ordered = sorted(rows, key=proxy_rank_key)
    if len(ordered) < quota:
        raise RuntimeError(f"quota={quota} cannot be satisfied; only {len(ordered)} rows available")
    grouped = defaultdict(list)
    for row in ordered:
        grouped[key_fn(row)].append(row)
    group_order = sorted(grouped)
    selected = []
    while len(selected) < quota and group_order:
        next_group_order = []
        for group_name in group_order:
            group_rows = grouped[group_name]
            if not group_rows:
                continue
            selected.append(group_rows.pop(0))
            if group_rows:
                next_group_order.append(group_name)
            if len(selected) >= quota:
                break
        group_order = next_group_order
    if len(selected) != quota:
        raise RuntimeError(f"quota={quota} was not filled; selected={len(selected)}")
    return selected


def desired_proxy_role_counts(total_rows, target_v3f, target_current):
    v3f_correct = round(target_v3f * total_rows)
    current_correct = round(target_current * total_rows)
    if abs((v3f_correct / total_rows) - target_v3f) > 1e-9:
        raise ValueError("target_v3f * total_rows must be an integer")
    if abs((current_correct / total_rows) - target_current) > 1e-9:
        raise ValueError("target_current * total_rows must be an integer")
    counts = {
        "both_correct": current_correct,
        "v3f_only": v3f_correct - current_correct,
        "current_only": 0,
        "both_wrong": total_rows - v3f_correct,
    }
    if counts["v3f_only"] < 0 or counts["both_wrong"] < 0:
        raise ValueError("invalid proxy target pair")
    return counts


def leaderboard_proxy_export_columns():
    return benchmark_columns() + [
        "_source_name",
        "_source_split",
        "proxy_role",
        "leaderboard_proxy_bucket",
        "leaderboard_proxy_note",
        "v1_focus_bucket",
        "v1_exposure_band",
        "v1_bucket_note",
        "v3f_prediction",
        "current_prediction",
        "v3f_is_correct",
        "current_is_correct",
    ]


def summarize_leaderboard_proxy_rows(rows, role_targets=None):
    total = len(rows)
    v3f_correct = sum(int(parse_bool(row.get("v3f_is_correct", False))) for row in rows)
    current_correct = sum(int(parse_bool(row.get("current_is_correct", False))) for row in rows)
    by_role = Counter(str(row.get("proxy_role", "")) for row in rows)
    by_source = Counter(str(row.get("_source_split", "")) for row in rows)
    by_family = Counter(str(row.get("family_short", "")) for row in rows)
    by_bucket = Counter(str(row.get("leaderboard_proxy_bucket", proxy_bucket(row))) for row in rows)
    return {
        "readme_eval_assumptions": {
            "metric": "accuracy",
            "temperature": README_TEMPERATURE,
            "top_p": README_TOP_P,
            "max_tokens": README_MAX_TOKENS,
            "max_num_seqs": README_MAX_NUM_SEQS,
            "max_model_len": README_MAX_MODEL_LEN,
            "boxed_first_extraction": True,
            "binary_answers_are_string_exact": True,
            "numeric_relative_tolerance": 1e-2,
        },
        "total_rows": total,
        "role_targets": role_targets or {},
        "selected_role_counts": dict(sorted(by_role.items())),
        "selected_source_split_counts": dict(sorted(by_source.items())),
        "selected_family_counts": dict(sorted(by_family.items())),
        "selected_bucket_counts": dict(sorted(by_bucket.items())),
        "proxy_scores": {
            "v3f": {
                "correct": v3f_correct,
                "rows": total,
                "accuracy": round(safe_div(v3f_correct, total), 4),
            },
            "current": {
                "correct": current_correct,
                "rows": total,
                "accuracy": round(safe_div(current_correct, total), 4),
            },
        },
    }


def render_leaderboard_proxy_report(summary):
    lines = []
    lines.append("# Leaderboard Proxy V1 Report")
    lines.append("")
    lines.append("## Purpose")
    lines.append("")
    lines.append("This benchmark is a local leaderboard proxy aligned to README.md inference conditions. It is engineered to reproduce the observed rank gap between v3f and the current proof-first route-aware run, not to claim recovery of the hidden leaderboard set.")
    lines.append("")
    lines.append("## Proxy scores")
    lines.append("")
    lines.append(f"- v3f: `{summary['proxy_scores']['v3f']['correct']} / {summary['proxy_scores']['v3f']['rows']} = {summary['proxy_scores']['v3f']['accuracy']:.4f}`")
    lines.append(f"- current: `{summary['proxy_scores']['current']['correct']} / {summary['proxy_scores']['current']['rows']} = {summary['proxy_scores']['current']['accuracy']:.4f}`")
    lines.append("")
    lines.append("## Selected role counts")
    lines.append("")
    lines.append("| role | rows |")
    lines.append("| --- | ---: |")
    for role_name, count in summary["selected_role_counts"].items():
        lines.append(f"| `{role_name}` | {count} |")
    lines.append("")
    lines.append("## Selected source splits")
    lines.append("")
    lines.append("| source_split | rows |")
    lines.append("| --- | ---: |")
    for source_name, count in summary["selected_source_split_counts"].items():
        lines.append(f"| `{source_name}` | {count} |")
    lines.append("")
    return "\n".join(lines)


def build_leaderboard_proxy_rows():
    source_rows = load_proxy_source_rows(SOURCE_PHASE0_SET_CSV, SOURCE_SPECIALIZED_SET_CSV)
    v3f_rows = {}
    v3f_rows.update(load_proxy_row_level_rows(V3F_PHASE0_ROW_LEVEL_CSV))
    v3f_rows.update(load_proxy_row_level_rows(V3F_SPECIALIZED_ROW_LEVEL_CSV))
    current_rows = {}
    current_rows.update(load_proxy_row_level_rows(CURRENT_PHASE0_ROW_LEVEL_CSV))
    current_rows.update(load_proxy_row_level_rows(CURRENT_SPECIALIZED_ROW_LEVEL_CSV))

    joined = []
    for row_id, source in source_rows.items():
        if row_id not in v3f_rows or row_id not in current_rows:
            continue
        v3f_row = v3f_rows[row_id]
        current_row = current_rows[row_id]
        row = dict(source)
        row["answer"] = source.get("answer", "")
        row["v3f_prediction"] = v3f_row["prediction"]
        row["current_prediction"] = current_row["prediction"]
        row["v3f_is_correct"] = bool(v3f_row["_corrected_is_correct"])
        row["current_is_correct"] = bool(current_row["_corrected_is_correct"])
        row["proxy_role"] = proxy_role_of(row["v3f_is_correct"], row["current_is_correct"])
        row["family_short"] = row.get("family_short") or normalize_family_label(row)
        joined.append(row)

    role_targets = desired_proxy_role_counts(
        LEADERBOARD_PROXY_TOTAL_ROWS,
        LEADERBOARD_PROXY_TARGET_V3F,
        LEADERBOARD_PROXY_TARGET_CURRENT,
    )
    pools = defaultdict(list)
    for row in joined:
        pools[row["proxy_role"]].append(row)

    selected = []
    for role_name in ("v3f_only", "both_correct", "both_wrong"):
        if role_name == "both_correct":
            key_fn = lambda row: f"{row.get('_source_split', '')}:{row.get('family_short', '')}" or "unknown"
        else:
            key_fn = proxy_bucket
        selected.extend(proxy_round_robin_take(pools[role_name], role_targets[role_name], key_fn))

    selected.sort(
        key=lambda row: (
            {"v3f_only": 0, "both_correct": 1, "both_wrong": 2}.get(row.get("proxy_role"), 9),
            row.get("_source_split", ""),
            proxy_bucket(row),
            -parse_float(row.get("hard_score"), 0.0),
            row.get("id", ""),
        )
    )
    for index, row in enumerate(selected, start=1):
        row["benchmark_name"] = LEADERBOARD_PROXY_NAME
        row["benchmark_role"] = "leaderboard_proxy"
        row["benchmark_index"] = index
        row["leaderboard_proxy_bucket"] = proxy_bucket(row)
        row["leaderboard_proxy_note"] = {
            "v3f_only": "Rows where v3f is correct and current is wrong; concentrates suspected hidden-regression slices.",
            "both_correct": "Rows both models solve; calibrates proxy topline upward without collapsing the gap.",
            "both_wrong": "Rows both models miss; approximates hidden hard tail not captured by local320.",
        }[row["proxy_role"]]

    return selected, summarize_leaderboard_proxy_rows(selected, role_targets)

In [ ]:
prebuilt_leaderboard_proxy_ready = os.path.exists(PREBUILT_LEADERBOARD_PROXY_DATASET_CSV)

if prebuilt_leaderboard_proxy_ready:
    print("Using prebuilt leaderboard proxy benchmark artifacts from Kaggle input")
    benchmark_rows = load_csv_rows(PREBUILT_LEADERBOARD_PROXY_DATASET_CSV)
    if os.path.exists(PREBUILT_LEADERBOARD_PROXY_SUMMARY_JSON):
        proxy_summary = json.loads(Path(PREBUILT_LEADERBOARD_PROXY_SUMMARY_JSON).read_text(encoding="utf-8"))
    else:
        proxy_summary = summarize_leaderboard_proxy_rows(benchmark_rows)
    source_mode = "prebuilt_proxy"
else:
    print("Prebuilt leaderboard proxy artifacts not found. Rebuilding from row-level artifacts...")
    benchmark_rows, proxy_summary = build_leaderboard_proxy_rows()
    source_mode = "rebuilt_from_row_level"

if "readme_eval_assumptions" not in proxy_summary:
    proxy_summary["readme_eval_assumptions"] = dict(
        proxy_summary.get(
            "readme_eval_contract",
            {
                "metric": "accuracy",
                "temperature": README_TEMPERATURE,
                "top_p": README_TOP_P,
                "max_tokens": README_MAX_TOKENS,
                "max_num_seqs": README_MAX_NUM_SEQS,
                "max_model_len": README_MAX_MODEL_LEN,
                "boxed_first_extraction": True,
                "binary_answers_are_string_exact": True,
                "numeric_relative_tolerance": 1e-2,
            },
        )
    )

manifest = {
    "phase": "leaderboard_proxy_eval",
    "source_mode": source_mode,
    "source_dataset_csv": PREBUILT_LEADERBOARD_PROXY_DATASET_CSV if prebuilt_leaderboard_proxy_ready else "rebuilt_in_notebook",
    "readme_eval_assumptions": proxy_summary["readme_eval_assumptions"],
    "proxy_summary": proxy_summary,
}

holdout_rows = []
benchmark_names = sorted({row["benchmark_name"] for row in benchmark_rows})

write_csv_rows(WORKING_COMBINED_BENCHMARK_CSV, benchmark_rows, leaderboard_proxy_export_columns())
write_json(WORKING_PROXY_SUMMARY_JSON, proxy_summary)
write_text(WORKING_PROXY_REPORT_MD, render_leaderboard_proxy_report(proxy_summary))
write_json(WORKING_MANIFEST_JSON, manifest)
write_text(WORKING_MANIFEST_MD, render_leaderboard_proxy_report(proxy_summary))

benchmark_df = pd.DataFrame(benchmark_rows)
print(f"Total Leaderboard Proxy rows: {len(benchmark_df)}")
display(benchmark_df.groupby(["benchmark_name", "family_short"]).size().reset_index(name="rows"))
display(pd.DataFrame([
    {"section": "role", "name": key, "rows": value}
    for key, value in proxy_summary["selected_role_counts"].items()
]))
display(pd.DataFrame([
    {"section": "source_split", "name": key, "rows": value}
    for key, value in proxy_summary["selected_source_split_counts"].items()
]))
print(render_leaderboard_proxy_report(proxy_summary))


In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=README_MAX_NUM_SEQS,
    gpu_memory_utilization=README_GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=README_MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=README_MAX_LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

sampling_params = SamplingParams(
    temperature=README_TEMPERATURE,
    top_p=README_TOP_P,
    max_tokens=README_MAX_TOKENS,
)

tokenizer = llm.get_tokenizer()
prompts = build_prompts(tokenizer, benchmark_df['prompt'].tolist())
outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest('adapter', 1, str(adapter_path)),
)

records = []
for row, output, rendered_prompt in zip(benchmark_rows, outputs, prompts):
    raw_output = output.outputs[0].text
    records.append({
        'benchmark_name': row['benchmark_name'],
        'benchmark_role': row['benchmark_role'],
        'benchmark_index': row['benchmark_index'],
        'family': row['family'],
        'family_short': row['family_short'],
        'template_subtype': row['template_subtype'],
        'selection_tier': row['selection_tier'],
        'teacher_solver_candidate': row['teacher_solver_candidate'],
        'answer_type': row['answer_type'],
        'num_examples': row['num_examples'],
        'prompt_len_chars': row['prompt_len_chars'],
        'id': row['id'],
        'expected_answer': row['answer'],
        'rendered_prompt': rendered_prompt,
        'raw_output': raw_output,
        'extracted_answer': extract_final_answer(raw_output),
    })

results_df = pd.DataFrame(records)
print(f"Generated {len(results_df)} Phase 0 samples with README-compatible vLLM settings")


In [ ]:
scored_rows = []
for row in records:
    derived = analyze_raw_output(row['raw_output'])
    prediction = derived['extracted_answer']
    scored_rows.append({
        'benchmark_name': row['benchmark_name'],
        'benchmark_role': row['benchmark_role'],
        'benchmark_index': row['benchmark_index'],
        'id': row['id'],
        'gold_answer': row['expected_answer'],
        'prediction': prediction,
        'is_correct': verify_answer(row['expected_answer'], prediction),
        'family': row['family'],
        'family_short': row['family_short'],
        'template_subtype': row['template_subtype'],
        'answer_type': row['answer_type'],
        'teacher_solver_candidate': row['teacher_solver_candidate'],
        'selection_tier': row['selection_tier'],
        'num_examples': row['num_examples'],
        'prompt_len_chars': row['prompt_len_chars'],
        'prompt_len_bucket': prompt_len_bucket(parse_int(row['prompt_len_chars'], 0)),
        'fallback_type': derived['fallback_type'],
        'format_bucket': derived['format_bucket'],
        'has_boxed': derived['has_boxed'],
        'boxed_count': derived['boxed_count'],
        'raw_output': row['raw_output'],
    })

row_level_df = pd.DataFrame(scored_rows)
overall_summary = summarize_scored_rows(scored_rows)
by_benchmark_summary = {
    benchmark_name: summarize_scored_rows(
        [row for row in scored_rows if row['benchmark_name'] == benchmark_name]
    )
    for benchmark_name in benchmark_names
}
binary_holdout_accuracy = build_binary_holdout_accuracy_rows(scored_rows, holdout_rows)

summary_payload = {
    'manifest': manifest,
    'overall': overall_summary,
    'by_benchmark': by_benchmark_summary,
    'binary_holdout_accuracy': binary_holdout_accuracy,
}

benchmark_accuracy_df = pd.DataFrame([
    {
        'benchmark_name': name,
        'rows': payload['overall']['rows'],
        'correct': payload['overall']['correct'],
        'accuracy': payload['overall']['accuracy'],
    }
    for name, payload in by_benchmark_summary.items()
])

display(benchmark_accuracy_df)
display(row_level_df[['benchmark_name', 'family_short', 'id', 'gold_answer', 'prediction', 'is_correct']].head(20))
display(pd.DataFrame(overall_summary['by_family']))
if binary_holdout_accuracy:
    display(pd.DataFrame(binary_holdout_accuracy))

print(render_markdown_summary('leaderboard_proxy_eval_overall', overall_summary))


In [ ]:
results_df.to_csv(WORKING_RAW_OUTPUTS_CSV, index=False)
row_level_df.to_csv(WORKING_ROW_LEVEL_CSV, index=False)
write_json(WORKING_SUMMARY_JSON, summary_payload)
write_text(WORKING_SUMMARY_MD, render_markdown_summary('leaderboard_proxy_eval_overall', overall_summary))

for benchmark_name, payload in by_benchmark_summary.items():
    benchmark_json = f"{WORKING_ARTIFACT_ROOT}/{benchmark_name}_summary.json"
    benchmark_md = f"{WORKING_REPORT_ROOT}/{benchmark_name}_summary.md"
    write_json(benchmark_json, payload)
    write_text(benchmark_md, render_markdown_summary(benchmark_name, payload))

preview_df = results_df.copy()
preview_df['prompt_preview'] = preview_df['rendered_prompt'].str.slice(0, 220)
preview_df['raw_output_preview'] = preview_df['raw_output'].str.slice(0, 600)
display(preview_df[['benchmark_name', 'family_short', 'id', 'prompt_preview', 'raw_output_preview']].head(20))

print(f"Saved raw outputs to {WORKING_RAW_OUTPUTS_CSV}")
print(f"Saved row-level scoring to {WORKING_ROW_LEVEL_CSV}")
print(f"Saved summary JSON to {WORKING_SUMMARY_JSON}")
print(f"Saved summary Markdown to {WORKING_SUMMARY_MD}")
print(f"Saved proxy benchmark CSV to {WORKING_COMBINED_BENCHMARK_CSV}")
print(f"Saved proxy summary JSON to {WORKING_PROXY_SUMMARY_JSON}")


## Notes

- `README.md` の評価仕様に合わせて、`\boxed{}` 優先抽出 + final-answer phrase fallback + last-number fallback + 数値相対誤差 `1e-2` で採点します。
- Kaggle 上では単一 LoRA を vLLM + `LoRARequest` で読み込み、submit 条件に近い推論設定のまま `leaderboard_proxy_v1_set` を採点します。
- `PREBUILT_LEADERBOARD_PROXY_ROOT` に proxy CSV / summary があればそれを優先し、無ければ v3f / current の row-level artifacts と source sets から notebook 内で benchmark を再構築します。
- この proxy は hidden leaderboard そのものではなく、v3f と current の既知の LB ギャップをローカル再現する engineered benchmark です。
- 出力は `/kaggle/working/leaderboard_proxy_eval/` 配下にまとまり、benchmark CSV / proxy summary / raw outputs / row-level / summary JSON / summary Markdown を保存します。
- adapter の Kaggle input path と fallback 用 artifact dataset path は環境ごとに変わるため、必要に応じて定数セルの path を更新してください。
